# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### Reflection: 
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv('https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/refs/heads/main/DataSets/hotels.csv')
print(df.shape)
print(df.isnull().sum()[df.isnull().sum() > 0])

(9404, 32)
country     266
agent      2335
company    8393
dtype: int64


I dropped columns that give away the answer or have too many blanks, turned the text columns into 0/1 columns, and then split the data 70/30.

In [2]:
df = df.drop(columns=['reservation_status', 'reservation_status_date', 'assigned_room_type', 'agent', 'company', 'country'])
df = pd.get_dummies(df, drop_first=True)

X = df.drop(columns=['is_canceled'])
y = df['is_canceled']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

(6582, 53) (2822, 53)


### ✍️ Your Response: 🔧
1. The dataset has 9,404 rows and 32 columns.

2. It has numerical columns like lead_time, adr, and number of nights, and categorical columns like meal, market_segment, deposit_type, and customer_type. is_canceled is the target and is 0 or 1.

3. Only country, agent, and company had missing values, so I dropped those columns instead of filling them in. I also dropped reservation_status, reservation_status_date, and assigned_room_type, because those are only known after the guest cancels or checks in, so they would give away the answer. Then I used get_dummies to turn the text columns into 0/1 columns and split the data 70/30.


## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.


In [3]:
nb = GaussianNB()
nb.fit(X_train, y_train)
nb_pred = nb.predict(X_test)

print(confusion_matrix(y_test, nb_pred))
print(classification_report(y_test, nb_pred))

[[ 910 1211]
 [  70  631]]
              precision    recall  f1-score   support

           0       0.93      0.43      0.59      2121
           1       0.34      0.90      0.50       701

    accuracy                           0.55      2822
   macro avg       0.64      0.66      0.54      2822
weighted avg       0.78      0.55      0.56      2822



### Reflection: 
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?

### ✍️ Your Response: 🔧
1. Not very well. Accuracy is only about 55%. It catches 90% of the cancellations (recall), but only 34% of the bookings it flags actually cancel (precision). Only about 25% of bookings cancel, so accuracy can be misleading. I think recall and precision for the canceled class are the best way to judge it.

2. Since it catches most cancellations, it could be a quick first check that flags bookings for staff to look at. It flags too many good bookings to use for anything automatic.


## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

In [4]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm = SVC(kernel='linear')
svm.fit(X_train_scaled, y_train)
svm_pred = svm.predict(X_test_scaled)

print(confusion_matrix(y_test, svm_pred))
print(classification_report(y_test, svm_pred))

[[2114    7]
 [ 412  289]]
              precision    recall  f1-score   support

           0       0.84      1.00      0.91      2121
           1       0.98      0.41      0.58       701

    accuracy                           0.85      2822
   macro avg       0.91      0.70      0.74      2822
weighted avg       0.87      0.85      0.83      2822



### Reflection: 
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?

### ✍️ Your Response: 🔧
1. Much better. Accuracy is about 85%. When it says a booking will cancel it is right 98% of the time (precision), but it only catches 41% of the cancellations (recall). Precision and recall for cancellations are still the best metrics, because accuracy looks good even if the model misses a lot of cancellations.

2. The SVM is good when the hotel needs to be sure before acting, like overbooking a room or calling a guest, because almost every booking it flags really does cancel.


## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn 
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

In [5]:
nn = MLPClassifier(hidden_layer_sizes=(16, 8), early_stopping=True, validation_fraction=0.2, max_iter=500, random_state=42)
nn.fit(X_train_scaled, y_train)
nn_pred = nn.predict(X_test_scaled)

print(confusion_matrix(y_test, nn_pred))
print(classification_report(y_test, nn_pred))

[[2022   99]
 [ 344  357]]
              precision    recall  f1-score   support

           0       0.85      0.95      0.90      2121
           1       0.78      0.51      0.62       701

    accuracy                           0.84      2822
   macro avg       0.82      0.73      0.76      2822
weighted avg       0.84      0.84      0.83      2822



### Reflection: 
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?

### ✍️ Your Response: 🔧
1. Its accuracy is about 84%, almost the same as the SVM and much better than Naive Bayes. It catches more cancellations than the SVM (51% recall vs 41%), but its precision is lower (78% vs 98%).

2. Probably not for big decisions. Managers can't see why it flags a booking, so it's hard to trust or explain to staff. It could be fine for something low risk, like deciding which guests get a reminder email.


## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

In [6]:
results = pd.Series({
    'Naive Bayes': accuracy_score(y_test, nb_pred),
    'SVM': accuracy_score(y_test, svm_pred),
    'Neural Network': accuracy_score(y_test, nn_pred)
})
results.sort_values(ascending=False)

SVM               0.851524
Neural Network    0.843019
Naive Bayes       0.546067
dtype: float64

### Reflection: 
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?

### ✍️ Your Response: 🔧
1. The SVM had the best accuracy at 85.2%, with the neural network close behind at 84.3%. Naive Bayes is the fastest to train and the easiest to use, but it only got 54.6%. The linear SVM is also easier to explain than the neural network, since it works like a weighted score for each feature.

2. Yes, I would deploy the SVM. It has the best accuracy, and when it flags a cancellation it's almost always right. The hotel should know it still misses more than half of the cancellations.


## 6. Final Business Recommendation

### Reflection: 
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?

2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. I recommend the SVM model. It predicts which bookings will cancel with about 85% accuracy, and when it flags a booking, it is right 98% of the time. That makes it useful for planning staff and deciding when it's safe to overbook. The main risk is that it misses more than half of the cancellations, so it shouldn't be the only tool. Data on past guest behavior, booking dates, and local events could improve it later.

2. One of my learning outcomes is a before and after case study for my dental clients using show rates and cost per lead. Predicting cancellations is a lot like predicting which patients won't show up. A model like this could flag patients who are likely to miss so the office can send extra reminders.


## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_12_bayes_svm_neural.ipynb"